## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
## add your code here
import sys

# 提高递归深度，防止边界用例下报错
sys.setrecursionlimit(300000)

class SubProblem:
    """
    分治策略的核心类：利用加法和异或魔法，将元素的低位特征进行对齐。
    """
    def __init__(self, elements, size):
        self.elements = elements
        self.size = size
        self.local_ops = []  # 记录当前子问题所需的魔法操作序列

    def solve(self):
        # 边界条件处理
        if self.size <= 0:
            return False
        
        # 检查元素是否合法（是否在 0 ~ size-1 范围内且无重复）
        visited = [False] * self.size
        for e in self.elements:
            if e >= self.size or e < 0:
                return False
            visited[e] = True
            
        if not all(visited):
            return False
            
        if self.size == 1:
            return True
            
        # 核心分治：按奇偶性拆分成左右两个子问题
        half = self.size // 2
        left_elements = [self.elements[i * 2] // 2 for i in range(half)]
        right_elements = [self.elements[i * 2 + 1] // 2 for i in range(half)]
        
        left_part = SubProblem(left_elements, half)
        right_part = SubProblem(right_elements, half)
        
        # 递归解决子问题
        if not left_part.solve() or not right_part.solve():
            return False
            
        # 调整首个元素的奇偶性对齐
        if self.elements[0] % 2 != 0:
            self.local_ops.append(1 if self.size == 2 else -1)
            
        # 合并左半部分的操作集，并累加异或影响 (xor_acc_A)
        xor_acc_A = 0
        for op in left_part.local_ops:
            if op > 0:
                self.local_ops.extend([-1, 1])
            else:
                self.local_ops.append(op * 2)
                xor_acc_A ^= (-op) * 2
                
        if xor_acc_A != 0:
            self.local_ops.append(-xor_acc_A)
            
        # 合并右半部分的操作集，并累加异或影响 (xor_acc_B)
        xor_acc_B = 0
        for op in right_part.local_ops:
            if op > 0:
                self.local_ops.extend([1, -1])
            else:
                self.local_ops.append(op * 2)
                xor_acc_B ^= (-op) * 2
                
        # 统一左右两侧的异或特征位
        if (xor_acc_B & half) != (xor_acc_A & half):
            for _ in range(self.size // 4):
                self.local_ops.extend([-1, 1])
                
        if xor_acc_A >= half: xor_acc_A -= half
        if xor_acc_B >= half: xor_acc_B -= half
        
        if xor_acc_A != xor_acc_B:
            return False
            
        # 操作序列优化：合并相邻的连续同类操作，抵消无效步骤
        optimized_ops = []
        for op in self.local_ops:
            if not optimized_ops:
                optimized_ops.append(op)
            else:
                if op < 0 and optimized_ops[-1] < 0:
                    combined = -((-optimized_ops[-1]) ^ (-op))
                    optimized_ops[-1] = combined
                    if optimized_ops[-1] == 0:
                        optimized_ops.pop()
                else:
                    optimized_ops.append(op)
                    
        self.local_ops = optimized_ops
        return True


def main():
    # 快速 I/O 读取输入
    input_data = sys.stdin.read().split()
    if not input_data:
        return
        
    N = int(input_data[0])
    swapA = int(input_data[1])
    swapB = int(input_data[2])
    
    arr = [int(x) for x in input_data[3:3+N]]
    final_operations = []
    
    # 寻找固定交换位置 (swapA, swapB) 的规律周期 (lowbit机制)
    K_offset = (swapA - swapB + N) % N
    K_offset &= -K_offset
    if K_offset == 0:
        K_offset = N

    # -- 辅助计算参数 --
    # 用于推导任意两点交换所需的异或与加法参数
    def calculate_params(targetA, targetB):
        diff = (targetB - targetA + N - K_offset + N) % N
        param1 = 0
        param2 = 0
        
        step = N // 2
        while step >= (K_offset * 2):
            if diff >= step:
                diff -= step
                param2 += step // 2
            else:
                param1 += step // 2
            step //= 2
            
        param1 += N // 2
        param1 += targetA & (K_offset - 1)
        param2 += targetA & (K_offset - 1)
        return param1, param2

    # 第一阶段：分治预处理，利用前面写好的 SubProblem 理顺数组的低位特征
    if K_offset > 1:
        init_elements = [arr[i] & (K_offset - 1) for i in range(min(N, K_offset))]
        problem = SubProblem(init_elements, K_offset)
        if not problem.solve():
            print("-1")
            return
            
        # 第一阶段是全局操作，使用列表推导式(C底层优化)极限加速
        for op in problem.local_ops:
            if op > 0:
                final_operations.append((2, op))  # 2代表加法魔法
                arr = [(x + op) % N for x in arr]
            else:
                final_operations.append((1, -op)) # 1代表异或魔法
                arr = [x ^ (-op) for x in arr]

    # 为了阶段二建立O(1)的值索引，极大降低时间复杂度
    # pos 数组记录了每个数字当前在数组中的位置
    pos = [0] * N
    for i, val in enumerate(arr):
        pos[val] = i

    # -- 极速版 O(1) 指令录制函数 --
    def record_add(val):
        if val != 0: final_operations.append((2, val))
    def record_xor(val):
        if val != 0: final_operations.append((1, val))
    def record_swap():
        final_operations.append((0, 0)) # 0代表交换魔法

    def swap_any(c, d):
        """核心优化：巧妙组合7步基础魔法，实现【任意两个位置】的宏观交换"""
        if (c // K_offset % 2) == (d // K_offset % 2):
            # 奇偶性相同，借助中间基准点进行三次桥接交换
            if (c // K_offset % 2) == 0:
                mid_base = (c & (K_offset - 1)) + K_offset
            else:
                mid_base = c & (K_offset - 1)
            swap_any(c, mid_base)
            swap_any(d, mid_base)
            swap_any(c, mid_base)
        else:
            # 奇偶性不同，直接通过异或和加法的精密配合（7步杀）进行调换
            pA, pB = calculate_params(swapA, swapB)
            pC, pD = calculate_params(c, d)
            
            # 记录那精妙的 7 步操作
            record_add((pC - c + N) % N)
            record_xor(pC ^ pA)
            record_add((swapA - pA + N) % N)
            
            record_swap()
            
            record_add((pA - swapA + N) % N)
            record_xor(pC ^ pA)
            record_add((c - pC + N) % N)
            
            # 【提速核心】物理上只通过 pos 直接去替换 c 和 d 两者的位置（O(1) 时间）
            # 不去真的遍历修改整个 arr 数组，节省大量时间
            idx_c, idx_d = pos[c], pos[d]
            arr[idx_c], arr[idx_d] = arr[idx_d], arr[idx_c]
            pos[c], pos[d] = idx_d, idx_c
                
    # 第二阶段：借助上面写好的任意交换能力（swap_any），像“环形排序”一样将数据最终归位
    for i in range(K_offset):
        group = []
        for j in range(i, N, K_offset):
            group.append(arr[j])
        group.sort()
        
        # 校验当前分组是否具有合法性（包含应有的数字）
        is_valid = True
        ptr = 0
        for j in range(i, N, K_offset):
            if group[ptr] != j:
                is_valid = False
                break
            ptr += 1
            
        if not is_valid:
            print("-1")
            return
            
        # 遍历归位：如果位置 j 上的数字不是 j，就把它和正确的数字换过来
        for j in range(i, N, K_offset):
            if arr[j] != j:
                # 只产生操作命令和做常数级别的数据交换，丝滑度拉满
                swap_any(j, arr[j])
                
    # 组装并打印最终的魔法使用步骤
    output = [str(len(final_operations))]
    for op_type, val in final_operations:
        if op_type == 0:
            output.append("0")
        else:
            output.append(f"{op_type} {val}")
            
    print('\n'.join(output))

if __name__ == '__main__':
    main()


## B 长跑

In [ ]:
## add your code here
import sys
from bisect import bisect_right

def solve():
    # 一次性读取所有输入，方便处理多组测试数据
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    
    idx = 0
    n_len = len(input_data)
    
    while idx < n_len:
        N = int(input_data[idx])
        L = int(input_data[idx+1])
        Maxn = int(input_data[idx+2])
        S = int(input_data[idx+3])
        idx += 4
        
        stations = []
        for _ in range(N):
            p = int(input_data[idx])
            c = int(input_data[idx+1])
            stations.append((p, c))
            idx += 2
        
        # 情况 1：不需要补给，初始体力可以直接到达终点
        if L <= Maxn:
            print("Yes")
            continue
            
        possible = False
        
        # 情况 2：只需 1 次补给
        for p, c in stations:
            # 判断条件：金币足够，能到达该站，且补给后能到达终点
            if c <= S and p <= Maxn and L - p <= Maxn:
                possible = True
                break
        
        if possible:
            print("Yes")
            continue
            
        # 情况 3：需要 2 次补给
        # 只有在总金币达到 2000 时，才有可能进行两次补给（每次最少花费 1000）
        if S == 2000:
            # 筛选出花费恰好为 1000 的补给站位置，去重并排序
            V = sorted(list(set(p for p, c in stations if c == 1000)))
            
            # 遍历第一次补给的站点 u
            for u in V:
                if u <= Maxn:
                    # 贪心策略：在能到达的范围内，寻找最远的第二次补给站点 v
                    # 即要求 v <= u + Maxn
                    pos = bisect_right(V, u + Maxn) - 1
                    if pos >= 0:
                        v = V[pos]
                        # 检查在最远的合法站点 v 补给后，能否完成剩余路程
                        if v >= L - Maxn:
                            possible = True
                            break
        
        # 输出当前测试用例的结果
        if possible:
            print("Yes")
        else:
            print("No")

if __name__ == '__main__':
    solve()


## C 最长回文

In [ ]:
import sys

def solve():
    # 快速读取全部输入，避免超时
    input_data = sys.stdin.read().split()
    if not input_data:
        return
        
    n = int(input_data[0])
    A = input_data[1]
    B = input_data[2]
    
    MOD = 999998639
    BASE = 13331
    
    # 1. 构建 Manacher 字符串，插入特殊字符处理偶数长度
    t_A = ["$", "#"]
    for c in A:
        t_A.append(c)
        t_A.append("#")
    proc_A = "".join(t_A)
    proc_n = len(proc_A)
    
    t_B = ["$", "#"]
    for c in B:
        t_B.append(c)
        t_B.append("#")
    proc_B = "".join(t_B)
    
    # 计算 Manacher 半径，展开内置 min 函数以优化常数时间
    def get_manacher_radius(s):
        p = [0] * proc_n
        mx, center_id = 0, 0
        for i in range(1, proc_n):
            if mx > i:
                mirror = 2 * center_id - i
                limit = mx - i
                p[i] = p[mirror] if p[mirror] < limit else limit
            else:
                p[i] = 1
                
            pi = p[i]
            while i + pi < proc_n and i - pi >= 0 and s[i + pi] == s[i - pi]:
                pi += 1
            p[i] = pi
            
            if i + pi > mx:
                mx = i + pi
                center_id = i
        return p
        
    p_A = get_manacher_radius(proc_A)
    p_B = get_manacher_radius(proc_B)

    # 2. 预处理哈希底数的幂
    base_pow = [1] * (proc_n + 5)
    for i in range(1, proc_n + 5):
        base_pow[i] = (base_pow[i - 1] * BASE) % MOD

    # 3. 错位哈希预处理，A正向，B反向，便于拼接对比
    hash_A = [0] * (proc_n + 1)
    for i in range(proc_n):
        hash_A[i + 1] = (hash_A[i] * BASE + ord(proc_A[i])) % MOD
        
    hash_B = [0] * (proc_n + 1)
    for i in range(proc_n - 1, -1, -1):
        hash_B[i] = (hash_B[i + 1] * BASE + ord(proc_B[i])) % MOD

    # 4. 核心逻辑：枚举拼接点寻找最长回文
    max_palindrome_len = 1
    
    for i in range(2, proc_n):
        rad_a = p_A[i]
        rad_b = p_B[i - 2] if i >= 2 else 0
        core_radius = rad_a if rad_a > rad_b else rad_b
        
        # 若基础半径能打破当前记录，优先更新
        if core_radius - 1 > max_palindrome_len:
            max_palindrome_len = core_radius - 1
            
        sa = i - core_radius + 1
        sb = i + core_radius - 3
        
        limit_left = sa
        limit_right = proc_n - 1 - sb
        max_limit = limit_left if limit_left < limit_right else limit_right
        
        # 计算打破当前记录所需达到的延伸长度
        req_ext = max_palindrome_len - core_radius + 2
        
        # 若剩余物理边界不足以打破记录，直接跳过
        if req_ext > max_limit:
            continue
            
        # 单次哈希碰撞校验：检查关键长度是否成立
        valA = (hash_A[sa] - hash_A[sa - req_ext] * base_pow[req_ext]) % MOD
        valB = (hash_B[sb + 1] - hash_B[sb + req_ext + 1] * base_pow[req_ext]) % MOD
        
        # 若不成立，说明无法打破记录，跳过本次循环
        if valA != valB:
            continue
            
        # 校验通过后，使用二分查找确定最大的延伸长度
        l = req_ext
        r = max_limit
        best_ext = req_ext
        
        while l <= r:
            mid = (l + r) >> 1
            vA = (hash_A[sa] - hash_A[sa - mid] * base_pow[mid]) % MOD
            vB = (hash_B[sb + 1] - hash_B[sb + mid + 1] * base_pow[mid]) % MOD
            if vA == vB:
                best_ext = mid
                l = mid + 1
            else:
                r = mid - 1
                
        # 计算新长度并更新最大回文串记录
        new_len = best_ext + core_radius - 1
        if new_len > max_palindrome_len:
            max_palindrome_len = new_len

    print(max_palindrome_len)

if __name__ == '__main__':
    solve()


## D 优惠券

In [ ]:
## add your code here
import sys
from bisect import bisect_left

def solve():
    # 一次性读取所有输入，提高处理大规模 I/O 的效率
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    
    iterator = iter(input_data)
    
    while True:
        try:
            m_str = next(iterator)
        except StopIteration:
            break
        
        m = int(m_str)
        
        if m == 0:
            print("-1")
            continue
        
        discards = []
        # parent 数组用于并查集，实现近似 O(1) 的状态跳跃
        # parent[i] 指向第 i 个 ? 之后，第一个未被使用的 ? 的索引
        parent = list(range(m + 1))
        
        # 迭代版本的并查集查找算法，包含路径压缩，避免递归过深
        def find(i):
            curr = i
            while parent[curr] != curr:
                curr = parent[curr]
            p = i
            while p != curr:
                nxt = parent[p]
                parent[p] = curr
                p = nxt
            return curr
        
        stat = {}  # 记录元素当前的持有状态
        last = {}  # 记录元素最后一次操作的行号
        
        early_exit_line = -1
        
        for l in range(m):
            op = next(iterator)
            if op == '?' or op == '？':
                discards.append(l)
            else:
                num = int(next(iterator))
                
                # 若已发现冲突行，继续遍历以消耗完本组输入数据，但不进行逻辑处理
                if early_exit_line != -1:
                    continue
                    
                cur_stat = stat.get(num, 0)
                cur_last = last.get(num, -1)  # 默认为 -1，确保首次匹配的下标大于等于 0
                
                if op == 'I':
                    # 如果当前已持有，再次执行 I 操作会产生冲突，需要将之前的某次 ? 转换为 O
                    if cur_stat == 1:
                        # 二分查找大于等于该元素上一次操作时间的 ? 的起始下标
                        idx = bisect_left(discards, cur_last)
                        # 使用并查集寻找下一个未被使用的 ?
                        real_idx = find(idx)
                        if real_idx >= len(discards):
                            early_exit_line = l + 1
                        else:
                            # 标记该 ? 已被使用，并更新其指向下一个位置
                            parent[real_idx] = real_idx + 1
                    stat[num] = 1
                    last[num] = l
                    
                elif op == 'O':
                    # 如果当前未持有，执行 O 操作会产生冲突，需要将之前的某次 ? 转换为 I
                    if cur_stat == 0:
                        idx = bisect_left(discards, cur_last)
                        real_idx = find(idx)
                        if real_idx >= len(discards):
                            early_exit_line = l + 1
                        else:
                            parent[real_idx] = real_idx + 1
                    stat[num] = 0
                    last[num] = l
        
        print(early_exit_line)

if __name__ == '__main__':
    solve()


## E 任意点

In [ ]:
## add your code here
import sys

# 增加递归深度限制，防止在特殊用例下递归层数过多引发错误
sys.setrecursionlimit(2000)

def solve():
    # 一次性读取所有输入数据，提高程序的读取效率
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    
    n = int(input_data[0])
    points = []
    
    idx = 1
    for _ in range(n):
        points.append((int(input_data[idx]), int(input_data[idx+1])))
        idx += 2
        
    visited = [False] * n
    
    # 定义深度优先搜索函数，遍历属于同一连通块的所有点
    def dfs(i):
        visited[i] = True
        for j in range(n):
            if not visited[j]:
                # 若两点的横坐标或纵坐标相同，则它们可以互相到达，属于同一连通块
                if points[i][0] == points[j][0] or points[i][1] == points[j][1]:
                    dfs(j)
    
    components = 0  # 记录连通块的总数
    
    for i in range(n):
        if not visited[i]:
            components += 1  # 发现一个新的连通块
            dfs(i)           # 遍历并标记该连通块内的所有关联点
            
    # 使所有点连通所需添加的最少点数，等于总连通块数量减 1
    print(components - 1)

if __name__ == '__main__':
    solve()


## F 通配符匹配
 


In [ ]:
## add your code here
#include <bits/stdc++.h>
#define ll long long
#define uint unsigned int
using namespace std;

char s[100004], rs[100004]; // s: 待匹配字符串(文件名), rs: 模式串(含通配符)
int p[14], cnt;             // p: 记录模式串中通配符的下标位置, cnt: 通配符总数
uint hs[100004], hsa[100004], p_pow[100004]; // hs/hsa: s和rs的前缀哈希数组, p_pow: 进制幂数组
int f[14][100004];          // f[i][j]: 模式串第i个通配符，匹配到文件名第j个字符是否合法 (1表示合法，0表示非法)

// 基于字符串哈希，O(1) 时间复杂度判断 s 的 [L1, R1] 区间与 rs 的 [L2, R2] 区间是否完全相等
bool check(int L1, int R1, int L2, int R2) {
    int N1, N2;
    // 计算待匹配串 s 在 [L1, R1] 的区间哈希值
    if (R1 >= L1) {
        N1 = (hs[R1] - ((hs[L1 - 1] * p_pow[R1 - L1 + 1])));
    } else {
        N1 = -1;
    }
    
    // 计算模式串 rs 在 [L2, R2] 的区间哈希值
    if (R2 >= L2) {
        N2 = (hsa[R2] - ((hsa[L2 - 1] * p_pow[R2 - L2 + 1])));
    } else {
        N2 = -1;
    }
    
    // 如果两段子串的哈希值相同，则认为它们相等
    return (N1 == N2);
}

int n;

void solve() {
    memset(f, 0, sizeof(f));
    cnt = 0;
    scanf("%d", &n); // 读取待匹配的文件名个数
    
    hsa[0] = 0;
    int lrs = strlen(rs + 1);
    
    // 找出模式串中所有 '*' 和 '?' 的位置，并记录到数组 p 中
    for (int i = 1; i <= lrs; i++) {
        if (rs[i] == '*' || rs[i] == '?') {
            p[++cnt] = i;
        }
    }
    
    // 在模式串末尾追加一个虚拟的 '?' 作为哨兵，方便处理边界
    p[++cnt] = lrs + 1;
    rs[++lrs] = '?';
    
    // 计算模式串 rs 的前缀哈希值（将 a-z 映射为 0-25，'*' 为 26，'?' 为 27，基数为 28）
    for (int i = 1; i <= lrs; i++) {
        int now;
        if ('a' <= rs[i] && rs[i] <= 'z') now = rs[i] - 'a';
        if (rs[i] == '*') now = 26;
        if (rs[i] == '?') now = 27;
        hsa[i] = (hsa[i - 1] * 28 + now);
    }
    
    // 依次处理 n 个文件名
    for (int CASE = 1; CASE <= n; CASE++) {
        scanf("%s", s + 1);
        int l = strlen(s + 1);
        
        // 初始化当前用例的 DP 状态数组
        for (int i = 0; i <= cnt; i++) {
            for (int j = 0; j <= l + 1; j++) {
                f[i][j] = 0;
            }
        }
            
        // 计算当前文件名 s 的前缀哈希值
        hs[0] = 0;
        for (int i = 1; i <= l; i++) {
            int now;
            if ('a' <= s[i] && s[i] <= 'z') now = s[i] - 'a';
            if (s[i] == '*') now = 26;
            if (s[i] == '?') now = 27;
            hs[i] = (hs[i - 1] * 28 + now);
        }
        
        // 在文件名末尾追加虚拟字符 '_' 作为哨兵对应模式串的虚拟哨兵
        s[++l] = '_';
        
        // DP 初始状态：0个通配符匹配0个字符是合法的
        f[0][0] = 1;
        
        // 开始状态转移：枚举每一个通配符
        for (int i = 0; i < cnt; i++) {
            // 如果当前通配符是 '*'，它可以匹配任意长度字符
            // 状态顺延：如果能匹配前 j-1 个字符，自然也能匹配前 j 个字符
            if (rs[p[i]] == '*') {
                for (int j = 1; j <= l; j++) {
                    if (f[i][j - 1]) f[i][j] = 1;
                }
            }
            
            // 枚举文件名中的位置 j，尝试向下一个通配符进行状态转移
            for (int j = 0; j <= l; j++) {
                if (!f[i][j]) continue; // 如果当前状态不可达，直接跳过
                
                // 检查：从当前位置向后，夹在两个通配符之间的普通文本段是否匹配
                // 长度为 p[i+1]-1 - (p[i]+1) + 1
                if (check(j + 1, j + (p[i + 1] - 1) - (p[i] + 1) + 1, p[i] + 1, p[i + 1] - 1)) {
                    // 如果下一个通配符是 '?'，它必须且只能匹配一个字符，状态向后推移 1 位
                    if (rs[p[i + 1]] == '?') {
                        f[i + 1][j + (p[i + 1] - 1) - (p[i] + 1) + 1 + 1] = 1;
                    } 
                    // 如果下一个通配符是 '*'，不需要多占字符，状态直接平移
                    else {
                        f[i + 1][j + (p[i + 1] - 1) - (p[i] + 1) + 1] = 1;
                    }
                }
            }
        }
        
        // 如果最后一个（哨兵）通配符能成功匹配到文件名的最后（哨兵处），说明整体匹配成功
        if (f[cnt][l]) puts("YES");
        else puts("NO");
    }
}

int main() {
    // 预处理哈希计算中需要用到的 28 进制次幂数组
    p_pow[0] = 1;
    for (int i = 1; i <= 100000; i++) {
        p_pow[i] = (p_pow[i - 1] * 28);
    }
    
    // 循环读取每一组模式串直到文件结束 (EOF)
    while (scanf("%s", rs + 1) != EOF) {
        solve();
    }
    return 0;
}


## G 汉诺塔



In [ ]:
## add your code here
import sys

def solve():
    # 读取所有输入数据
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    
    n = int(input_data[0])
    priorities = input_data[1:7]
    
    # 建立映射
    peg_map = {'A': 0, 'B': 1, 'C': 2}
    
    # priority_rank[u][v] 存储移动操作 u->v 的优先级排名
    # 排名越小，优先级越高
    priority_rank = [[-1] * 3 for _ in range(3)]
    for i, move in enumerate(priorities):
        u = peg_map[move[0]]
        v = peg_map[move[1]]
        priority_rank[u][v] = i
        
    # dp[i][u] 表示将 i 个盘子构成的塔，从 u 移动完成的最少步数
    dp = [[0] * 3 for _ in range(n + 1)]
    
    # nxt[i][u] 表示将 i 个盘子构成的塔，从 u 移动完毕最终到达的柱子编号
    nxt = [[0] * 3 for _ in range(n + 1)]
    
    # 初始化 base case: 只有一个盘子 (i=1)
    for u in range(3):
        best_v = -1
        best_rank = float('inf')
        # 寻找优先级最高的合法目的地
        for v in range(3):
            if u != v and priority_rank[u][v] < best_rank:
                best_rank = priority_rank[u][v]
                best_v = v
        nxt[1][u] = best_v
        dp[1][u] = 1
        
    # 状态转移: i 从 2 到 n
    for i in range(2, n + 1):
        for u in range(3):
            # 第一阶段: i-1 个盘子从 u 移到 v
            v = nxt[i-1][u]
            w = 3 - u - v  # w 是剩余的那根空闲柱子 (0+1+2=3)
            
            # 此时最大的盘子被移到 w
            # i-1 个盘子接下来从 v 出发会移向哪里 
            x = nxt[i-1][v]
            
            if x == w:
                # 刚好与最大盘子在 w 汇合
                dp[i][u] = dp[i-1][u] + 1 + dp[i-1][v]
                nxt[i][u] = w
            elif x == u:
                # 小盘子跑回了原点 u，大盘子在 w
                # 把大盘子从 w 移到 v，然后小盘子再从 u 移到 v 汇合
                dp[i][u] = 2 * dp[i-1][u] + dp[i-1][v] + 2
                nxt[i][u] = v
                
    # 输出将 n 个盘子从 A 柱移走的总步数
    print(dp[n][0])

if __name__ == '__main__':
    solve()


## H 马步距离

In [ ]:
## add your code here
import sys

def solve():
    # 读取所有输入数据
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    
    # 解析四个坐标点
    xp, yp, xs, ys = map(int, input_data)
    
    # 计算起点到终点在 x 和 y 方向的相对绝对位移
    dx = abs(xp - xs)
    dy = abs(yp - ys)
    
    # 利用对称性，始终让 x 为较大的偏移量，y 为较小的偏移量
    x = max(dx, dy)
    y = min(dx, dy)
    
    # 特判两个无法通过通用公式计算的近端死角情况
    if x == 1 and y == 0:
        print(3)
        return
    if x == 2 and y == 2:
        print(4)
        return
        
    # 计算基础最小步数 等价于max(ceil(x/2), ceil((x+y)/3))
    res = max((x + 1) // 2, (x + y + 2) // 3)
    
    # 马每走一步，横纵坐标变化量之和(2+1=3)是奇数 所以最终步数的奇偶性必须和终点坐标和的奇偶性保持一致
    if res % 2 != (x + y) % 2:
        res += 1
        
    print(res)

if __name__ == '__main__':
    solve()


## I 直方图最大矩形

In [ ]:
## add your code here
class Solution:
    def largestRectangleArea(self, heights: list[int]) -> int:
        # 在尾部追加一个高度为 0 的虚拟柱子，强制最后清空栈并计算所有剩余面积
        heights.append(0) 
        
        stack = []    # 单调递增栈，存储的是 heights 的索引
        max_area = 0  # 记录全局最大面积
        
        for i in range(len(heights)):
            # 当遇到比栈顶矮的柱子，破坏了单调递增性，说明栈顶柱子的“右边界”找到了
            while stack and heights[i] < heights[stack[-1]]:
                # 1. 确定高度
                h = heights[stack.pop()]
                
                # 2. 确定宽度
                # 如果栈为空，说明弹出的柱子左边没有比它更矮的，宽度可延伸至索引 0
                # 如果栈不为空，左边界就是当前栈顶元素的索引（不包含），右边界是 i（不包含）
                w = i if not stack else i - stack[-1] - 1
                
                # 3. 更新最大面积
                max_area = max(max_area, h * w)
            
            # 保持单调递增，当前索引入栈
            stack.append(i)
            
        return max_area


## J 消防局的设立

In [ ]:
## add your code here
import sys
from collections import deque

def solve():
    # 优化 I/O 读取速度，一次性读取所有输入
    input_data = sys.stdin.read().split()
    if not input_data:
        return
        
    n = int(input_data[0])
    if n == 0:
        print(0)
        return
        
    adj = [[] for _ in range(n + 1)]
    
    # 构建树，根据题意，第 i 行的数据代表 i 和 a[i] 之间有一条边
    # input_data[1] 是 a[2], input_data[2] 是 a[3] ...
    for i in range(2, n + 1):
        p = int(input_data[i - 1])
        adj[p].append(i)
        adj[i].append(p)
        
    # BFS 获取层序遍历序列，保证自底向上的处理顺序
    order = []
    parent = [0] * (n + 1)
    vis = [False] * (n + 1)
    
    q = deque([1])
    vis[1] = True
    
    while q:
        u = q.popleft()
        order.append(u)
        for v in adj[u]:
            if not vis[v]:
                vis[v] = True
                parent[v] = u
                q.append(v)
                
    # 状态初始化
    req = [0] * (n + 1)            # 最远未覆盖节点的距离，初始为 0
    dp = [float('inf')] * (n + 1)  # 最近消防局的距离，初始为无穷大
    ans = 0
    
    # 逆序遍历 BFS 序列，即自底向上处理
    for i in range(n - 1, -1, -1):
        u = order[i]
        
        # 1. 互相抵消：子树内的消防局能覆盖到最远的未覆盖节点
        if req[u] + dp[u] <= 2:
            req[u] = -float('inf')
            
        # 2. 被迫建站：最远未覆盖节点距离达到 2，必须建站
        if req[u] == 2:
            ans += 1
            req[u] = -float('inf')
            dp[u] = 0
            
        # 3. 向上传递状态
        if u != 1:
            p = parent[u]
            req[p] = max(req[p], req[u] + 1)
            dp[p] = min(dp[p], dp[u] + 1)
            
    # 检查根节点是否仍有未被覆盖的节点
    if req[1] >= 0:
        ans += 1
        
    print(ans)

if __name__ == '__main__':
    solve()
